# Spam
# Simple Baselines using ``mean`` and ``last`` pooling

## Librairies

In [2]:
# !pip install transformers==4.8.2
# !pip install datasets==1.7.0
# !pip install ax-platform==0.1.20

In [3]:
import os
import sys
sys.path.insert(0, os.path.abspath("../..")) # comment this if library is pip installed

In [4]:
import io
import re
import pickle
from timeit import default_timer as timer

from tqdm.notebook import tqdm

import numpy as np
import random

import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import AutoTokenizer
from transformers import BertModel
from transformers.data.data_collator import DataCollatorWithPadding

from ax import optimize
from ax.plot.contour import plot_contour
from ax.plot.trace import optimization_trace_single_method
from ax.service.managed_loop import optimize
from ax.utils.notebook.plotting import render, init_notebook_plotting

import esntorch.core.reservoir as res
import esntorch.core.learning_algo as la
import esntorch.core.pooling_strategy as ps
import esntorch.core.esn as esn

In [5]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [7]:
# SEED = 42

## Global variables

In [8]:
CACHE_DIR = '~/Data/huggignface/' # put your path here
RESULTS_FILE = 'Results/Baselines_v2/spam_results_.pkl' # put your path here

## Dataset

In [9]:
# rename correct column as 'labels': depends on the dataset you load

def tokenize(sample):
    """Tokenize sample"""
    
    sample = tokenizer(sample['Text'], truncation=True, padding=False, return_length=True)
    
    return sample
    
def load_and_enrich_dataset(dataset_name, split, cache_dir):
    """
    Load dataset from the datasets library of HuggingFace.
    Tokenize and add length.
    """
    
    # Load dataset
    dataset = load_dataset(dataset_name, split=split, cache_dir=CACHE_DIR)
    
    # Rename label column for tokenization purposes (use 'label-fine' for fine-grained labels)
    dataset = dataset.rename_column('Label', 'labels')
    
    # Tokenize data
    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.rename_column('length', 'lengths')
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels', 'lengths'])
    
    return dataset

In [10]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# train set
# full_train_dataset = load_and_enrich_dataset('Mustafaege/sms-spam-balanced', split='train', cache_dir=CACHE_DIR).sort("lengths")

# mini train and val sets: 70%/30% of 30%
# XXX here we increase the train_size split since small dataset
train_dataset = load_and_enrich_dataset('Mustafaege/sms-spam-balanced', split='train', cache_dir=CACHE_DIR).sort("lengths")
val_dataset = load_and_enrich_dataset('Mustafaege/sms-spam-balanced', split='validation', cache_dir=CACHE_DIR).sort("lengths")
full_train_dataset = concatenate_datasets([train_dataset, val_dataset]).sort("lengths")

# test set
test_dataset = load_and_enrich_dataset('Mustafaege/sms-spam-balanced', split='test', cache_dir=CACHE_DIR).sort("lengths")

dataset_d = {
    'full_train': full_train_dataset,
    'train': train_dataset,
    'val': val_dataset,
    'test': test_dataset
    }

# dataloaders
dataloader_d = {}
for k, v in dataset_d.items():
    dataloader_d[k] = torch.utils.data.DataLoader(v, batch_size=256, collate_fn=DataCollatorWithPadding(tokenizer))

Generating train split:   0%|          | 0/1045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/149 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1045 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [11]:
dataset_d

{'full_train': Dataset({
     features: ['labels', 'Text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 1194
 }),
 'train': Dataset({
     features: ['labels', 'Text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 1045
 }),
 'val': Dataset({
     features: ['labels', 'Text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 149
 }),
 'test': Dataset({
     features: ['labels', 'Text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 300
 })}

## Optimization

In [12]:
baseline_params = {
                'embedding': 'bert-base-uncased', # TEXT.vocab.vectors,
                'distribution' : 'uniform',               # uniform, gaussian
                'input_dim' : 768,                        # dim of encoding!
                'reservoir_dim' : 0,      # not used
                'bias_scaling' : 0.0,     # not used
                'sparsity' : 0.0,         # not used
                'spectral_radius' : None, 
                'leaking_rate': 0.5,      # not used
                'activation_function' : 'tanh',
                'input_scaling' : 0.1,
                'mean' : 0.0,
                'std' : 1.0,
                'learning_algo' : None,
                'criterion' : None,
                'optimizer' : None,
                'pooling_strategy' : None,
                'lexicon' : None,
                'bidirectional' : False,
                'mode' : 'no_layer',     # simple baseline
                'device' : device,
                'seed' : random.randint(0, 10000)
                }

In [13]:
results_d = {}

for pooling_strategy in tqdm(['last', 'mean']):
    
    results_d[pooling_strategy] = {}
    
    for alpha in tqdm([0.1, 1.0, 10.0, 100.0]):
        
        results_d[pooling_strategy][alpha] = []
        
        # model
        baseline_params['pooling_strategy'] = pooling_strategy
        baseline_params['mode'] = 'no_layer'
        print(baseline_params)
        ESN = esn.EchoStateNetwork(**baseline_params)
        ESN.learning_algo = la.RidgeRegression(alpha=alpha)
        ESN = ESN.to(device)

        # train
        t0 = timer()
        LOSS = ESN.fit(dataloader_d["full_train"])                       # full train set
        t1 = timer()
        acc = ESN.predict(dataloader_d["test"], verbose=False)[1] # full test set

        if hasattr(acc, "item"):
            acc = acc.item()

        # results
        results_d[pooling_strategy][alpha].append([acc, t1 - t0])
        
        # clean objects
        del ESN.learning_algo
        del ESN.criterion
        del ESN.pooling_strategy
        del ESN
        torch.cuda.empty_cache()

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

{'embedding': 'bert-base-uncased', 'distribution': 'uniform', 'input_dim': 768, 'reservoir_dim': 0, 'bias_scaling': 0.0, 'sparsity': 0.0, 'spectral_radius': None, 'leaking_rate': 0.5, 'activation_function': 'tanh', 'input_scaling': 0.1, 'mean': 0.0, 'std': 1.0, 'learning_algo': None, 'criterion': None, 'optimizer': None, 'pooling_strategy': 'last', 'lexicon': None, 'bidirectional': False, 'mode': 'no_layer', 'device': device(type='cpu'), 'seed': 4490}
Model downloaded: bert-base-uncased
Computing closed-form solution...


  0%|          | 0/5 [00:00<?, ?it/s]


Training complete.


...

  0%|          | 0/2 [00:00<?, ?it/s]

{'embedding': 'bert-base-uncased', 'distribution': 'uniform', 'input_dim': 768, 'reservoir_dim': 0, 'bias_scaling': 0.0, 'sparsity': 0.0, 'spectral_radius': None, 'leaking_rate': 0.5, 'activation_function': 'tanh', 'input_scaling': 0.1, 'mean': 0.0, 'std': 1.0, 'learning_algo': None, 'criterion': None, 'optimizer': None, 'pooling_strategy': 'mean', 'lexicon': None, 'bidirectional': False, 'mode': 'no_layer', 'device': device(type='cpu'), 'seed': 4490}
Model downloaded: bert-base-uncased
Computing closed-form solution...


  0%|          | 0/5 [00:00<?, ?it/s]


Training complete.


  0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
results_d

{'last': {0.1: [[88.66666666666667, 115.74392799999998]],
  1.0: [[90.33333333333333, 111.45684980000001]],
  10.0: [[94.33333333333333, 105.59150470000003]],
  100.0: [[93.66666666666667, 100.47762169999999]]},
 'mean': {0.1: [[86.0, 99.2967324]],
  1.0: [[90.66666666666667, 93.31940960000009]],
  10.0: [[94.0, 98.06866589999993]],
  100.0: [[95.66666666666667, 95.27858289999995]]}}

## Results

In [15]:
# save results

with open(RESULTS_FILE, 'wb') as fh:
    pickle.dump(results_d, fh)

FileNotFoundError: [Errno 2] No such file or directory: 'Results/Baselines_v2/spam_results_.pkl'

In [ ]:
# # load results
# with open(os.path.join(RESULTS_PATH, RESULTS_FILE), 'rb') as fh:
#     results_d = pickle.load(fh)

In [ ]:
# results_d